# Demo 2: Error Scenarios (Non-Technical Walkthrough)\n\n## Objective\nShow how the API protects data and business rules:\n1. Rejects missing credentials (`401`)\n2. Rejects orders without stock (`409`)\n3. Rejects invalid payload (`422`)\n\n## Before you start\nRun these commands in terminal: \n- `make up`\n- `make health`

In [ ]:
BASE_URL = "http://localhost:8000"\nBASE_URL

In [ ]:
import json\nimport uuid\n\nimport requests\n\ndef show_response(label, response):\n    print(f"\n=== {label} ===")\n    print("Status:", response.status_code)\n    try:\n        print(json.dumps(response.json(), indent=2, ensure_ascii=False))\n    except Exception:\n        print(response.text)

## Setup: create user, login, and low-stock product

In [ ]:
run_id = uuid.uuid4().hex[:8]\nemail = f"demo_error_{run_id}@example.com"\npassword = "StrongPass123"\n\npayload = {"email": email, "password": password}\nrequests.post(f"{BASE_URL}/auth/register", json=payload, timeout=15)\nr_login = requests.post(f"{BASE_URL}/auth/login", json=payload, timeout=15)\nshow_response("Login", r_login)\ntoken = r_login.json().get("access_token")\nheaders = {"Authorization": f"Bearer {token}"}\n\nproduct_id = f"low_{run_id}"\nproduct_payload = {\n    "product_id": product_id,\n    "product_name": "Low Stock Product",\n    "quantity": 1\n}\nr_create = requests.post(f"{BASE_URL}/products", json=product_payload, timeout=15)\nshow_response("Create Low-Stock Product", r_create)

## 1) Missing Credentials Case\nExpected: **401** with `no valid credentials`.

In [ ]:
r_no_auth = requests.post(\n    f"{BASE_URL}/orders",\n    json={"product_id": product_id, "quantity": 1},\n    timeout=15,\n)\nshow_response("Order Without Credentials", r_no_auth)

## 2) No Stock Case\nWe first consume the single available unit, then try one more order.\nExpected second call: **409** with `no_stock`.

In [ ]:
r_first_order = requests.post(\n    f"{BASE_URL}/orders",\n    json={"product_id": product_id, "quantity": 1},\n    headers=headers,\n    timeout=15,\n)\nshow_response("First Valid Order (consumes stock)", r_first_order)\n\nr_no_stock = requests.post(\n    f"{BASE_URL}/orders",\n    json={"product_id": product_id, "quantity": 1},\n    headers=headers,\n    timeout=15,\n)\nshow_response("Second Order (no stock)", r_no_stock)

## 3) Invalid Payload Case\nExpected: **422** for invalid quantity (negative number).

In [ ]:
r_invalid = requests.post(\n    f"{BASE_URL}/orders",\n    json={"product_id": product_id, "quantity": -3},\n    headers=headers,\n    timeout=15,\n)\nshow_response("Invalid Payload", r_invalid)

## Non-Technical Interpretation\n- Unauthorized users cannot place orders.\n- The system blocks overselling.\n- Bad input is rejected before affecting data.